# Assignment 03

Build a machine learning model that predicts the fair market price of a used car from its attributes (make, model, year, kilometers driven, fuel type, transmission, ownership history, engine specs, location, etc.).

Data label: fair market price

Data features:
* Make
* Model
* Year
* Kilometers driven
* Fuel type
* Transmission
* Ownership history
* Engine specs
* Location
* Drivetrain
* Seating capacity
* Fuel tank capacity

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

In [2]:
df = pd.read_csv('car details v4.csv')

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2059 entries, 0 to 2058
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Make                2059 non-null   str    
 1   Model               2059 non-null   str    
 2   Price               2059 non-null   int64  
 3   Year                2059 non-null   int64  
 4   Kilometer           2059 non-null   int64  
 5   Fuel Type           2059 non-null   str    
 6   Transmission        2059 non-null   str    
 7   Location            2059 non-null   str    
 8   Color               2059 non-null   str    
 9   Owner               2059 non-null   str    
 10  Seller Type         2059 non-null   str    
 11  Engine              1979 non-null   str    
 12  Max Power           1979 non-null   str    
 13  Max Torque          1979 non-null   str    
 14  Drivetrain          1923 non-null   str    
 15  Length              1995 non-null   float64
 16  Width            

Determine if/how many null entries are present

In [3]:
df.isna().sum()

Make                    0
Model                   0
Price                   0
Year                    0
Kilometer               0
Fuel Type               0
Transmission            0
Location                0
Color                   0
Owner                   0
Seller Type             0
Engine                 80
Max Power              80
Max Torque             80
Drivetrain            136
Length                 64
Width                  64
Height                 64
Seating Capacity       64
Fuel Tank Capacity    113
dtype: int64

In [4]:
# Try to recover NA data from same model vehicles.

model_to_engine_map = (
    df.dropna(subset=['Engine'])
    .groupby('Model')['Engine']
    .agg(lambda s: s.mode()[0])
    .to_dict()
)

model_to_max_power_map = (
    df.dropna(subset=['Max Power'])
    .groupby('Model')['Max Power']
    .agg(lambda s: s.mode()[0])
    .to_dict()
)

model_to_drivetrain_map = (
    df.dropna(subset=['Drivetrain'])
    .groupby('Model')['Drivetrain']
    .agg(lambda s: s.mode()[0])
    .to_dict()
)

model_to_length_map = (
    df.dropna(subset=['Length'])
    .groupby('Model')['Length']
    .agg(lambda s: s.mode()[0])
    .to_dict()
)

model_to_width_map = (
    df.dropna(subset=['Width'])
    .groupby('Model')['Width']
    .agg(lambda s: s.mode()[0])
    .to_dict()
)

model_to_height_map = (
    df.dropna(subset=['Height'])
    .groupby('Model')['Height']
    .agg(lambda s: s.mode()[0])
    .to_dict()
)

model_to_seating_map = (
    df.dropna(subset=['Seating Capacity'])
    .groupby('Model')['Seating Capacity']
    .agg(lambda s: s.mode()[0])
    .to_dict()
)

model_to_fuel_tank_map = (
    df.dropna(subset=['Fuel Tank Capacity'])
    .groupby('Model')['Fuel Tank Capacity']
    .agg(lambda s: s.mode()[0])
    .to_dict()
)

df['Engine'] = df['Engine'].fillna(df['Model'].map(model_to_engine_map))
df['Max Power'] = df['Max Power'].fillna(df['Model'].map(model_to_max_power_map))
df['Drivetrain'] = df['Drivetrain'].fillna(df['Model'].map(model_to_drivetrain_map))
df['Length'] = df['Length'].fillna(df['Model'].map(model_to_length_map))
df['Width'] = df['Width'].fillna(df['Model'].map(model_to_width_map))
df['Height'] = df['Height'].fillna(df['Model'].map(model_to_height_map))
df['Seating Capacity'] = df['Seating Capacity'].fillna(df['Model'].map(model_to_seating_map))
df['Fuel Tank Capacity'] = df['Fuel Tank Capacity'].fillna(df['Model'].map(model_to_fuel_tank_map))


In [5]:
df.isna().sum()

Make                    0
Model                   0
Price                   0
Year                    0
Kilometer               0
Fuel Type               0
Transmission            0
Location                0
Color                   0
Owner                   0
Seller Type             0
Engine                 68
Max Power              68
Max Torque             80
Drivetrain            118
Length                 52
Width                  52
Height                 52
Seating Capacity       52
Fuel Tank Capacity     95
dtype: int64

We have clearly made some headway here. As for the rest, they'll get dropped. These are all things a buyer would care about when purchasing a car and shouldn't be interpolated from existing data.

In [6]:
df.dropna(inplace=True)

df.info()

<class 'pandas.DataFrame'>
Index: 1887 entries, 0 to 2057
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Make                1887 non-null   str    
 1   Model               1887 non-null   str    
 2   Price               1887 non-null   int64  
 3   Year                1887 non-null   int64  
 4   Kilometer           1887 non-null   int64  
 5   Fuel Type           1887 non-null   str    
 6   Transmission        1887 non-null   str    
 7   Location            1887 non-null   str    
 8   Color               1887 non-null   str    
 9   Owner               1887 non-null   str    
 10  Seller Type         1887 non-null   str    
 11  Engine              1887 non-null   str    
 12  Max Power           1887 non-null   str    
 13  Max Torque          1887 non-null   str    
 14  Drivetrain          1887 non-null   str    
 15  Length              1887 non-null   float64
 16  Width               18

Splitting Max Power and Max Torque

| Max Power | Max Power BHP | Max Power RPM |
| --- | --- | ---|
| 87 bhp @ 6000 rpm | 87 | 6000 |



| Max Torque | Max Torque Nm | Max Torque RPM |
| --- | --- | ---|
| 74 Nm @ 4000 rpm | 74 | 4000 |

In [7]:
# Split max power 

df[['Max Power BHP', 'Max Power RPM']] = df['Max Power'].str.split('@', n=1, expand=True)
df['Max Power BHP'] = df['Max Power BHP'].str.strip()
df['Max Power RPM'] = df['Max Power RPM'].str.strip()

In [8]:
# Split max torque

df[['Max Torque Nm', 'Max Torque RPM']] = df['Max Power'].str.split('@', n=1, expand=True)
df['Max Torque Nm'] = df['Max Torque Nm'].str.strip()
df['Max Torque RPM'] = df['Max Torque RPM'].str.strip()

In [9]:
df.info()

<class 'pandas.DataFrame'>
Index: 1887 entries, 0 to 2057
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Make                1887 non-null   str    
 1   Model               1887 non-null   str    
 2   Price               1887 non-null   int64  
 3   Year                1887 non-null   int64  
 4   Kilometer           1887 non-null   int64  
 5   Fuel Type           1887 non-null   str    
 6   Transmission        1887 non-null   str    
 7   Location            1887 non-null   str    
 8   Color               1887 non-null   str    
 9   Owner               1887 non-null   str    
 10  Seller Type         1887 non-null   str    
 11  Engine              1887 non-null   str    
 12  Max Power           1887 non-null   str    
 13  Max Torque          1887 non-null   str    
 14  Drivetrain          1887 non-null   str    
 15  Length              1887 non-null   float64
 16  Width               18

In [10]:
# remove units for columns with them
df['Engine'] = df['Engine'].astype(str).str.replace(r'[^\d.]', '', regex=True)
df['Max Power BHP'] = df['Max Power BHP'].astype(str).str.replace(r'[^\d.]', '', regex=True)
df['Max Power RPM'] = df['Max Power RPM'].astype(str).str.replace(r'[^\d.]', '', regex=True)
df['Max Torque Nm'] = df['Max Torque Nm'].astype(str).str.replace(r'[^\d.]', '', regex=True)
df['Max Torque RPM'] = df['Max Torque RPM'].astype(str).str.replace(r'[^\d.]', '', regex=True)

In [11]:
# Turn into numbers
df[['Max Power BHP', 'Max Power RPM', 'Max Torque Nm', 'Max Torque RPM', 'Engine']] = df[['Max Power BHP', 'Max Power RPM', 'Max Torque Nm', 'Max Torque RPM', 'Engine']].apply(pd.to_numeric, errors='coerce')

In [12]:
df.info()

<class 'pandas.DataFrame'>
Index: 1887 entries, 0 to 2057
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Make                1887 non-null   str    
 1   Model               1887 non-null   str    
 2   Price               1887 non-null   int64  
 3   Year                1887 non-null   int64  
 4   Kilometer           1887 non-null   int64  
 5   Fuel Type           1887 non-null   str    
 6   Transmission        1887 non-null   str    
 7   Location            1887 non-null   str    
 8   Color               1887 non-null   str    
 9   Owner               1887 non-null   str    
 10  Seller Type         1887 non-null   str    
 11  Engine              1887 non-null   int64  
 12  Max Power           1887 non-null   str    
 13  Max Torque          1887 non-null   str    
 14  Drivetrain          1887 non-null   str    
 15  Length              1887 non-null   float64
 16  Width               18

In [13]:
df.nunique()

Make                   32
Model                 956
Price                 587
Year                   18
Kilometer             793
Fuel Type               7
Transmission            2
Location               75
Color                  16
Owner                   5
Seller Type             3
Engine                104
Max Power             311
Max Torque            266
Drivetrain              3
Length                235
Width                 163
Height                189
Seating Capacity        6
Fuel Tank Capacity     55
Max Power BHP         157
Max Power RPM          40
Max Torque Nm         157
Max Torque RPM         40
dtype: int64

In [14]:
# Before getting dummies, try to get same model, different trim cars together

two_word_models = {
    'Grand':{'i10', 'Vitara'},
    'Elite':{'i20'},
    'Vitara':{'Brezza'},
    'Corolla':{'Altis'},
    'Range':{'Rover'},
    'Santa':{'Fe'},
    'Wagon':{'R'}
}

def base_model(model):
    tokens = model.split()
    if len(tokens) >= 2 and tokens[1] in two_word_models.get(tokens[0], set()):
        return f'{tokens[0]} {tokens[1]}'
    return tokens[0]

df['Model Base'] = df['Model'].apply(base_model)

In [16]:
df.nunique()

Make                   32
Model                 956
Price                 587
Year                   18
Kilometer             793
Fuel Type               7
Transmission            2
Location               75
Color                  16
Owner                   5
Seller Type             3
Engine                104
Max Power             311
Max Torque            266
Drivetrain              3
Length                235
Width                 163
Height                189
Seating Capacity        6
Fuel Tank Capacity     55
Max Power BHP         157
Max Power RPM          40
Max Torque Nm         157
Max Torque RPM         40
Model Base            182
dtype: int64

In [ ]:
if 'Model Base' not in df.columns:
    df['Model Base'] = df['Model'].apply(base_model)

price_means = df.groupby('Model Base')['Price'].mean()

KeyError: 'Model Base'

In [18]:
categories = ['Make', 'Base Model', 'Fuel Type', 'Transmission', 'Location', 'Color', 'Owner', 'Seller Type', 'Drivetrain']

df = pd.get_dummies(
    df,
    columns=['Make', 'Model Base', 'Fuel Type', 'Transmission', 'Location', 'Color', 'Owner', 'Seller Type', 'Drivetrain'],
    drop_first=True,
    dtype=int
)

df.dropna(inplace=True)

df.info()

<class 'pandas.DataFrame'>
Index: 1885 entries, 0 to 2057
Columns: 332 entries, Model to Drivetrain_RWD
dtypes: float64(9), int64(320), str(3)
memory usage: 4.8 MB


In [87]:
cols_to_drop = ['Price', 'Max Power', 'Max Torque']

X = df.drop(columns=cols_to_drop)
y = df['Price']

Now We can start building the model.

In [88]:
from sklearn.model_selection import train_test_split

(X_train, X_test, y_train, y_test) = train_test_split(X, y, test_size=0.2, random_state=36)

In [89]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, root_mean_squared_error, mean_absolute_error

model = LinearRegression()
model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](1102,)","[161017.94, -1.53, -831.42,...,-28785.34,105990.39,301421.84]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](1102,)","['Year','Kilometer','Engine',...,'Seller Type_Individual','Drivetrain_FWD', 'Drivetrain_RWD']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,-3.216e+08
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,1102
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(106)
